In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

In [3]:
df = pd.read_csv('BTC-USD.csv')

In [5]:
df.head(10)

,Date,Open,High,Low,Close,Adj Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,398.821014,26580100
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,402.152008,24127600
6,2014-09-23,402.092010,441.557007,396.196991,435.790985,435.790985,45099500
7,2014-09-24,435.751007,436.112000,421.131989,423.204987,423.204987,30627700
8,2014-09-25,423.156006,423.519989,409.467987,411.574005,411.574005,26814400
9,2014-09-26,411.428986,414.937988,400.009003,404.424988,404.424988,21460800


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3520 entries, 0 to 3519
Data columns (total 7 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   Date       3520 non-null   object 
 1   Open       3520 non-null   float64
 2   High       3520 non-null   float64
 3   Low        3520 non-null   float64
 4   Close      3520 non-null   float64
 5   Adj Close  3520 non-null   float64
 6   Volume     3520 non-null   int64  
dtypes: float64(5), int64(1), object(1)
memory usage: 192.6+ KB


In [7]:
df.isnull().sum()

Date         0
Open         0
High         0
Low          0
Close        0
Adj Close    0
Volume       0
dtype: int64

In [10]:
df.drop(columns=['Adj Close'], inplace= True)

In [11]:
df.sort_index(inplace=True)

In [12]:
df.head(10)

,Date,Open,High,Low,Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,24127600
6,2014-09-23,402.092010,441.557007,396.196991,435.790985,45099500
7,2014-09-24,435.751007,436.112000,421.131989,423.204987,30627700
8,2014-09-25,423.156006,423.519989,409.467987,411.574005,26814400
9,2014-09-26,411.428986,414.937988,400.009003,404.424988,21460800


### Stationarity checking of the dataset

In [13]:
from statsmodels.tsa.stattools import adfuller

In [16]:
def run_adf_test(Series):
    result = adfuller(Series.dropna())

    print(f'ADF Statistic : {result[0]:.4f}')
    print(f'P value : {result[1]:.4f}')

    if result[1] <= 0.05:
       print("Result: Stationary (Reject H0)")
    else:
        print("Result: Non-Stationary (Fail to reject H0)")

print("--- Raw BTC Prices ---")
run_adf_test(df['Close'])


--- Raw BTC Prices ---
ADF Statistic : -0.8982
P value : 0.7886
Result: Non-Stationary (Fail to reject H0)


In [17]:
print("\n--- Log Returns ---")
log_returns = np.log(df['Close'] / df['Close'].shift(1))
run_adf_test(log_returns)


--- Log Returns ---
ADF Statistic : -60.5543
P value : 0.0000
Result: Stationary (Reject H0)


In [18]:
df.head(10)

,Date,Open,High,Low,Close,Volume
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100
5,2014-09-22,399.100006,406.915985,397.130005,402.152008,24127600
6,2014-09-23,402.092010,441.557007,396.196991,435.790985,45099500
7,2014-09-24,435.751007,436.112000,421.131989,423.204987,30627700
8,2014-09-25,423.156006,423.519989,409.467987,411.574005,26814400
9,2014-09-26,411.428986,414.937988,400.009003,404.424988,21460800


In [23]:
df['Log_Returns'] = log_returns.dropna()

In [24]:
df.head()

,Date,Open,High,Low,Close,Volume,Log_Returns
0,2014-09-17,465.864014,468.174011,452.421997,457.334015,21056800,NaN
1,2014-09-18,456.859985,456.859985,413.104004,424.440002,34483200,-0.074643
2,2014-09-19,424.102997,427.834991,384.532013,394.795990,37919700,-0.072402
3,2014-09-20,394.673004,423.295990,389.882996,408.903992,36863600,0.035111
4,2014-09-21,408.084991,412.425995,393.181000,398.821014,26580100,-0.024968


In [25]:
# Create a series that only contains the valid numbers
clean_log_returns = df['Log_Returns'].dropna()

# Now the ADF test will work perfectly
result = adfuller(clean_log_returns)

In [28]:
df = df.dropna()

### Outlier checking

In [31]:
mean = df['Log_Returns'].mean()
std = df['Log_Returns'].std()

# Define outliers as anything 3 standard deviations away
df.loc[:, 'Is_Outlier'] = (df['Log_Returns'] > mean + 3*std) | (df['Log_Returns'] < mean - 3*std)

# See how many outliers you found
print(df['Is_Outlier'].sum())

57


In [46]:
df.head()

,Date,Open,High,Low,Close,Volume,Log_Returns,Is_Outlier,Target_Classification,Log_Returns_Lag_1,Volume_Lag_1,Log_Returns_Lag_2,Volume_Lag_2,Log_Returns_Lag_3,Volume_Lag_3,Volatility_7d
13,2014-09-30,376.088013,390.976990,373.442993,386.944000,34707300,0.030109,False,0,-0.004555,32497700.0,-0.057539,23613300.0,-0.012202,15029300.0,0.026803
14,2014-10-01,387.427002,391.378998,380.779999,383.614990,26229400,-0.008641,False,0,0.030109,34707300.0,-0.004555,32497700.0,-0.057539,23613300.0,0.026354
15,2014-10-02,383.988007,385.497009,372.946014,375.071991,21777700,-0.022521,False,0,-0.008641,26229400.0,0.030109,34707300.0,-0.004555,32497700.0,0.025960
16,2014-10-03,375.181000,377.695007,357.859009,359.511993,30901200,-0.042370,False,0,-0.022521,21777700.0,-0.008641,26229400.0,0.030109,34707300.0,0.028238
17,2014-10-04,359.891998,364.487000,325.885986,328.865997,47236500,-0.089097,False,0,-0.042370,30901200.0,-0.022521,21777700.0,-0.008641,26229400.0,0.039036


In [34]:
from sklearn.preprocessing import RobustScaler

features = ['Log_Returns', 'Volume']
X = df[features]

# 2. Time-Series Split (e.g., 80% training, 20% testing)
split_idx = int(len(X) * 0.8)
train_df = X.iloc[:split_idx]
test_df = X.iloc[split_idx:]

# 3. Fit the scaler ONLY on the training data to prevent leakage
scaler = RobustScaler()
scaler.fit(train_df)

# 4. Transform both training and testing sets
X_train_scaled = scaler.transform(train_df)
X_test_scaled = scaler.transform(test_df)

print("Scaling complete. Median of scaled training data is now:", X_train_scaled.mean(axis=0))

Scaling complete. Median of scaled training data is now: [-0.01570373  0.37271274]


### Here we use XGBoost classifier for predicting market direction (Up/Down), 
### Because we are dealing with a time-series dataset, we must ensure our target arrays (y_train, y_test) align perfectly with the chronological split we performed during the scaling step.

### 1. Prepare Targets and Align the SplitFirst, map out the labels and create the exact same train/test split index used for your scaled features.

In [38]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Create the binary target (1 = Up, 0 = Down/Flat for the NEXT day)
df.loc[:,'Target_Classification'] = (df['Log_Returns'].shift(-1) > 0).astype(int)

# Drop the last row since it has no "tomorrow" to predict
df_clean = df.dropna(subset=['Target_Classification'])

# Re-run your exact split index based on the cleaned data
split_idx = int(len(df_clean) * 0.8)

# Align the target arrays with your scaled features
# (Assuming X_train_scaled and X_test_scaled have been sliced up to split_idx)
y_train = df_clean['Target_Classification'].iloc[:split_idx].values
y_test = df_clean['Target_Classification'].iloc[split_idx:].values

### 2. Train the XGBoost ModelWhen dealing with financial data, standard trees overfit quickly. We add regularization parameters like max_depth (keeping trees shallow) and learning_rate to prevent the model from memorizing noise.

In [39]:
# 2. Initialize the model with conservative parameters for finance
model = XGBClassifier(
    n_estimators=100,
    max_depth=3,            # Shallow trees limit overfitting to market noise
    learning_rate=0.05,     # Small steps yield more stable models
    random_state=42,
    eval_metric='logloss'
)

# 3. Train the model
model.fit(X_train_scaled, y_train)

,objective,'binary:logistic'
,use_label_encoder,False
,base_score,0.5
,booster,'gbtree'
,callbacks,None
,colsample_bylevel,1
,colsample_bynode,1
,colsample_bytree,1
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,'logloss'


### 3. Evaluate the Results: In crypto trading, raw accuracy can be deceptive. We evaluate using Precision (out of all the times the model predicted "UP", how many times did the price actually go up?) to avoid false buy signals.

In [40]:
# 4. Predict on the test set
y_pred = model.predict(X_test_scaled)

# 5. Print metrics
print("--- Model Performance ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))

--- Model Performance ---
Accuracy Score: 0.4872

Classification Report:
              precision    recall  f1-score   support

   Down/Flat       0.49      0.16      0.24       358
          Up       0.49      0.82      0.61       346

    accuracy                           0.49       704
   macro avg       0.49      0.49      0.43       704
weighted avg       0.49      0.49      0.42       704



## An accuracy score of 48.72% means your model is currently performing slightly worse than a random coin flip (50%).Looking closely at your classification report, your model has fallen into a classic time-series trap: it is predicting "Up" almost all the time.

## Why did this happen?
### High Recall for 'Up' (0.82): The model predicted "Up" for 82% of the actual upward days.
### Low Recall for 'Down' (0.16): The model only successfully caught 16% of the downward days.
### The Cause: Because Bitcoin historically has a strong upward bias, the model learned that the easiest way to minimize its error during training was to just guess "Up" over and over again. It is failing to find actual patterns in your features (Log_Returns and Volume).

## 1. Add "Lagged" Features (Crucial)
### Right now, your model only looks at today's return and today's volume to predict tomorrow. It has no concept of a trend. You need to give it a window into the past by adding lags:

In [45]:
# Create lag features for the past 3 days
for lag in [1, 2, 3]:
    df.loc[:,f'Log_Returns_Lag_{lag}'] = df['Log_Returns'].shift(lag)
    df.loc[:,f'Volume_Lag_{lag}'] = df['Volume'].shift(lag)

# Add a volatility metric (7-day rolling standard deviation)
df['Volatility_7d'] = df['Log_Returns'].rolling(window=7).std()

df = df.dropna()

In [58]:
# Added new features for the training data set 
features = ['Log_Returns','Log_Returns_Lag_1','Volume_Lag_1','Log_Returns_Lag_2', 'Volume_Lag_2', 'Log_Returns_Lag_3', 'Volume_Lag_3', 'Volatility_7d']
X = df[features]

y = df['Target_Classification'].values

# 5. Time-Series Chronological Split (80% Train, 20% Test)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

# 6. Fit the Scaler strictly on Training Data
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete. Median of scaled training data is now:", X_train_scaled.mean(axis=0))

Scaling complete. Median of scaled training data is now: [-0.01450238 -0.0139375   0.37174552 -0.01429922  0.37202253 -0.01463754
  0.37197279  0.17923691]


## 2. Address Class Imbalance:
### Since the model is heavily biased toward predicting "Up", you can force XGBoost to penalize errors on "Down" days more heavily.Calculate the ratio of negative days to positive days in your training set.Add scale_pos_weight to your classifier:

In [59]:
# Calculate balance: total_negative_examples / total_positive_examples
negative_cases = (y_train == 0).sum()

positive_cases = (y_train == 1).sum()

ratio = negative_cases / positive_cases
model = XGBClassifier(
    scale_pos_weight=ratio,  # Forces the model to pay attention to Down days
    max_depth=2,             # Lower depth to reduce overfitting to noise
    learning_rate=0.01,
    n_estimators=100
)

model.fit(X_train_scaled, y_train)

,objective,'binary:logistic'
,use_label_encoder,False
,base_score,0.5
,booster,'gbtree'
,callbacks,None
,colsample_bylevel,1
,colsample_bynode,1
,colsample_bytree,1
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


## New Model predictions :

In [60]:
# Evaluate updated predictions
y_pred = model.predict(X_test_scaled)
print(f"New Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(classification_report(y_test, y_pred, target_names=['Down/Flat', 'Up']))

New Accuracy Score: 0.5142
              precision    recall  f1-score   support

   Down/Flat       0.53      0.47      0.50       358
          Up       0.50      0.56      0.53       344

    accuracy                           0.51       702
   macro avg       0.52      0.52      0.51       702
weighted avg       0.52      0.51      0.51       702



## Feature importance findings:

In [61]:

# 1. Extract importance scores from the trained model
importance_scores = model.feature_importances_

# 2. Pair them with your feature names
feature_importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importance_scores
}).sort_values(by='Importance', ascending=False)

# 3. Print the sorted rankings
print("--- XGBoost Feature Importance Rankings ---")
print(feature_importance_df.to_string(index=False))

--- XGBoost Feature Importance Rankings ---
          Feature  Importance
Log_Returns_Lag_2    0.163486
     Volume_Lag_1    0.156993
    Volatility_7d    0.139314
      Log_Returns    0.133897
     Volume_Lag_2    0.115276
Log_Returns_Lag_3    0.108124
Log_Returns_Lag_1    0.097113
     Volume_Lag_3    0.085797


# Price Prediction Regression:

In [63]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# 1. Drop 'Volume' from your features list as it added 0 value
optimized_features = [
    'Log_Returns', 'Log_Returns_Lag_2', 'Volume_Lag_1', 
    'Volatility_7d', 'Volume_Lag_2', 'Log_Returns_Lag_3', 
    'Log_Returns_Lag_1', 'Volume_Lag_3'
]

X_opt = df[optimized_features]

# 2. Define the REGRESSION target (The actual next-day log return value)
y_reg = df['Log_Returns'].shift(-1).dropna()
X_opt = X_opt.iloc[:-1] # Align X by dropping the last row

# 3. Time-Series Chronological Split (80% Train, 20% Test)
split_idx = int(len(X_opt) * 0.8)
X_train, X_test = X_opt.iloc[:split_idx], X_opt.iloc[split_idx:]
y_train, y_test = y_reg.iloc[:split_idx], y_reg.iloc[split_idx:]

# 4. Scale your optimized features
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 5. Train the XGBoost Regressor
reg_model = XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.03,
    random_state=42,
    eval_metric='mae' # Mean Absolute Error optimization
)
reg_model.fit(X_train_scaled, y_train)

# 6. Predict and Evaluate
y_pred = reg_model.predict(X_test_scaled)
print("--- Regressor Performance ---")
print(f"Mean Absolute Error (MAE): {mean_absolute_error(y_test, y_pred):.6f}")
print(f"R-squared (R2) Score: {r2_score(y_test, y_pred):.4f}")

--- Regressor Performance ---
Mean Absolute Error (MAE): 0.029583
R-squared (R2) Score: -0.7451


## Your model is heavily overfitting—it is chasing the noise of the training data and making wild, incorrect predictions on the test set.

### Why is the Regressor struggling?
### Lags are too short for value extraction: While a 2-day lag can tell you if the market went up or down, it doesn't give a model enough structural       context to guess if Bitcoin will move by $200 or $5,000.
### XGBoost Regressors default to overfitting: Trees are excellent at structural thresholds, but they are notorious for failing at continuous sequence     extrapolation in financial time series.

In [64]:

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.preprocessing import RobustScaler

# 1. Prepare your optimized features and regression target
optimized_features = [
    'Log_Returns', 'Log_Returns_Lag_2', 'Volume_Lag_1', 
    'Volatility_7d', 'Volume_Lag_2', 'Log_Returns_Lag_3', 
    'Log_Returns_Lag_1', 'Volume_Lag_3'
]

# X is our feature matrix, y is the actual next-day continuous log return
X_raw = df[optimized_features].values
y_raw = df['Log_Returns'].shift(-1).dropna().values
X_raw = X_raw[:-1] # Align lengths

# 2. Scale the data chronologically
split_idx = int(len(X_raw) * 0.8)
scaler = RobustScaler()

X_train_scaled = scaler.fit_transform(X_raw[:split_idx])
X_test_scaled = scaler.transform(X_raw[split_idx:])
y_train, y_test = y_raw[:split_idx], y_raw[split_idx:]

# 3. Create the 3D Sliding Window Function
def create_3d_windows(X_data, y_data, lookback=10):
    X_3d, y_3d = [], []
    for i in range(len(X_data) - lookback):
        X_3d.append(X_data[i : i + lookback]) # Take rows from i to i+lookback
        y_3d.append(y_data[i + lookback])     # Predict the target at the end of the window
    return np.array(X_3d), np.array(y_3d)

# Convert arrays into 3D shapes (Samples, 10 Time Steps, 8 Features)
LOOKBACK_DAYS = 10
X_train_3d, y_train_3d = create_3d_windows(X_train_scaled, y_train, lookback=LOOKBACK_DAYS)
X_test_3d, y_test_3d = create_3d_windows(X_test_scaled, y_test, lookback=LOOKBACK_DAYS)

# 4. Build the LSTM Architecture
model_lstm = Sequential([
    # First LSTM layer requires input_shape = (time_steps, features)
    LSTM(units=50, return_sequences=False, input_shape=(X_train_3d.shape[1], X_train_3d.shape[2])),
    Dropout(0.2), # Dropout layer prevents overfitting to noise
    Dense(units=25, activation='relu'),
    Dense(units=1) # Single output node for continuous price return forecasting
])

model_lstm.compile(optimizer='adam', loss='mean_squared_error')

# 5. Train the Network
history = model_lstm.fit(
    X_train_3d, y_train_3d, 
    epochs=20, 
    batch_size=32, 
    validation_data=(X_test_3d, y_test_3d),
    verbose=1
)


C:\Users\Karnulu Suresh\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 5s 15ms/step - loss: 0.0047 - val_loss: 8.3815e-04
Epoch 2/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0022 - val_loss: 7.5443e-04
Epoch 3/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0019 - val_loss: 7.6674e-04
Epoch 4/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0018 - val_loss: 7.2260e-04
Epoch 5/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0017 - val_loss: 7.3818e-04
Epoch 6/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0016 - val_loss: 7.2591e-04
Epoch 7/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0016 - val_loss: 7.2606e-04
Epoch 8/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0016 - val_loss: 7.1675e-04
Epoch 9/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.0016 - val_loss: 7.1622e-04
Epoch 10/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 0.0016 - val_loss: 7.1394e-04
Epoch 11/20
88/88 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - loss: 0.0015 - val_loss: 7.1551e-04
Epoch 12/20
88/88 ━━━━━━━

In [65]:
y_pred_lstm = model_lstm.predict(X_test_3d)

print(f"LSTM MAE: {mean_absolute_error(y_test_3d, y_pred_lstm):.6f}")
print(f"LSTM R2 Score: {r2_score(y_test_3d, y_pred_lstm):.4f}")

22/22 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step
LSTM MAE: 0.017945
LSTM R2 Score: 0.0028


In [70]:
import os
import pickle

# 1. Create directory if missing
os.makedirs('models', exist_ok=True)

# 2. Save RobustScaler
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ RobustScaler saved to models/scaler.pkl")

# 3. Save XGBoost Classifier using PICKLE (Bypasses Python 3.13 JSON bug)
# (Assuming your variable name is 'model' from the training script)
with open('models/classifier.pkl', 'wb') as f:
    pickle.dump(model, f)
print("✅ XGBoost Classifier saved safely to models/classifier.pkl")

# 4. Save Keras LSTM Regressor
# (Assuming your variable name is 'model_lstm')
model_lstm.save('models/regressor_lstm.h5')
print("✅ LSTM Regressor saved to models/regressor_lstm.h5")


✅ RobustScaler saved to models/scaler.pkl
✅ XGBoost Classifier saved safely to models/classifier.pkl
✅ LSTM Regressor saved to models/regressor_lstm.h5
